# Part 3 RAG logic

In this notebook, you will build a complete **Retrieval-Augmented Generation (RAG)** system that answers questions using the HKU Innovation Wing and InnoAcademy knowledge base you created earlier.

### What you will learn
- How to securely load API keys from a `.env` file
- How to connect to the ChromaDB vector database
- How to perform **semantic retrieval** (finding the most relevant chunks)
- How to build a clean prompt and generate accurate answers with GPT-4o-mini
- Best practices for context formatting and source citation

By the end, you will have a working RAG pipeline that can intelligently answer questions about Innovation Wing programs, workshops, projects, and events.

## 3.1. Introduction to Retrieval-Augmented Generation (RAG)

Even the best large language models can hallucinate or lack up-to-date knowledge.  

**RAG solves this** by:
1. **Retrieving** the most relevant documents from your vector database
2. **Augmenting** the prompt with this retrieved context
3. **Generating** an accurate, grounded answer using the LLM

This notebook combines the embeddings you created in Part 2 with GPT-4o-mini to build a reliable question-answering system for the HKU Innovation Wing and InnoAcademy.

## 3.2. Install and Import Libraries

We use:
- **`openai`** (Azure) → for both embeddings and chat completions
- **`chromadb`** → to query the vector database
- **`python-dotenv`** → to securely load API keys

Run the cell below to install any missing packages.

In [1]:
%pip install openai chromadb

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os
import chromadb
from openai import AzureOpenAI
from pathlib import Path
BASE_DIR = Path.cwd().parent

## 3.3. Set Up Your Azure OpenAI API Key (Important!)
If you have already set up your API key, skip this step

**Step-by-step instructions:**
1. Create a new file named **`.env`** in the **same folder** as this notebook (if you haven’t already).
2. Open the `.env` file and add the following line:

```env
AZURE_OPENAI_API_KEY=your_actual_azure_openai_api_key_here
```

3. Save the file. The notebook will automatically load your key using `python-dotenv`.


>**Tip**: The `.env` file should start with a dot and must be in the same directory as `3_rag.ipynb`.  
>Never share or upload this file to GitHub.

## 3.4. Setup OpenAI Clients

We create two Azure OpenAI clients:
- One for **embeddings** (used during retrieval)
- One for **chat completions** (GPT-4o-mini for answer generation)

The code will check if your API key is loaded correctly from the `.env` file.

In [10]:
load_dotenv(BASE_DIR / ".env")

API_Key = os.getenv("AZURE_OPENAI_API_KEY")

if not API_Key:
    raise RuntimeError("Missing Azure OpenAI credentials. Set AZURE_OPENAI_API_KEY in .env or environment.")

embedding_client = AzureOpenAI(
    azure_endpoint="https://api-iw.azure-api.net/sig-embedding/openai/deployments/text-embedding-3-small/embeddings?api-version=2024-10-21",
    api_key=API_Key,
    api_version="2024-10-21",
)

GPT_client = AzureOpenAI(
    azure_endpoint="https://api-iw.azure-api.net/sig-shared-jpeast/deployments/gpt-4o-mini/chat/completions?api-version=2025-01-01-preview",
    api_key=API_Key,
    api_version="2025-01-01-preview",
)

## 3.5. Connect to ChromaDB Vector Database

We connect to the persistent ChromaDB created in the previous notebook.

Make sure you have already run the embedding generator notebook (`2_embedding_generator.ipynb`) so that the collection `Innowing_db` exists.

In [11]:
CHROMA_PATH = str(BASE_DIR / (os.getenv("CHROMA_PATH") or "chroma_db/chroma_db"))
COLLECTION_NAME = "Innowing_db"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection(name=COLLECTION_NAME)

## 3.6. Define the Question

You can change the `question` variable below to ask anything about the Innovation Wing or InnoAcademy.

Try questions like:
- “What workshops are available?”
- “Tell me about the Robot Arm Challenge”
- “What is the Innovation Academy?”

In [12]:
question = "What is the Innovation Academy?"

## 3.7. Perform Semantic Retrieval

This is the **retrieval** step of RAG:
- Convert the user question into an embedding
- Search the vector database for the top `k` most similar chunks (semantic search)
- Return the text, URL, and similarity score of each retrieved document

We use `top_k = 5` by default. You can adjust this value to retrieve more or fewer context documents.

In [13]:
top_k = 5

if not question.strip():
    results = collection.peek(limit=top_k)
    context_docs = [{"text": doc, "url": meta.get("url", ""), "score": 1.0}
            for doc, meta in zip(results["documents"], results["metadatas"])]

response = embedding_client.embeddings.create(
    input=question.replace("\n", " "),
    model=os.getenv("EMBEDDING_MODEL", "text-embedding-3-small"),
)
query_embedding = response.data[0].embedding

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=top_k,
    include=["documents", "metadatas", "distances"]
)

context_docs = []
for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    # Convert distance to similarity score (Chroma uses cosine distance by default)
    score = 1 - distance
    context_docs.append({
        "text": doc,
        "score": round(score, 4)
    })

Let’s print the retrieved documents so we can see what information will be given to the LLM.

This step is very useful for debugging and understanding how good (or limited) the retrieval quality is.

In [14]:
print("Retrieved Context Documents:")
for idx, doc in enumerate(context_docs, 1):
    print(f"{idx}. {doc['text'][:100]}... (Score: {doc['score']})")

Retrieved Context Documents:
1. About us – Innovation Academy
Skip to content
Main Menu
Home
About us
Programmes and activities
Menu... (Score: 0.65)
2. Innovation Academy
Skip to content
Main Menu
Home
About us
Programmes and activities
Menu Toggle
Wor... (Score: 0.5917)
3. Innovation Academy
Skip to content
Main Menu
Home
About us
Programmes and activities
Menu Toggle
Wor... (Score: 0.5917)
4. Pitching – Innovation Academy
Skip to content
Main Menu
Home
About us
Programmes and activities
Menu... (Score: 0.5838)
5. Sharing – Innovation Academy
Skip to content
Main Menu
Home
About us
Programmes and activities
Menu ... (Score: 0.5551)


## 3.8. Augment the retrieved trunks

Now we perform the **augmentation** step:
- Combine the retrieved context documents into a single prompt

This grounded approach greatly reduces hallucinations.

In [15]:
context_parts = []
for i, doc in enumerate(context_docs, 1):
        context_parts.append(
            f"Document {i} :\n{doc['text']}"
        )

context = "\n\n---\n\n".join(context_parts)

In [16]:
print("\nConstructed Context for LLM:")
print(context)


Constructed Context for LLM:
Document 1 :
About us – Innovation Academy
Skip to content
Main Menu
Home
About us
Programmes and activities
Menu Toggle
Workshop
Sharing
Pitching
Student-initiated courses
Study Tour
Inno Show and Carnivals
Robot Arm Challenge 2026
Student Development Projects
Funding Scheme
Internship Opportunities
Innovation Wing
About us
HKU Faculty of Engineering is committed to fostering contemporary engineering education for students to contribute to the industry and addressing the changing needs of the community. In keeping with this commitment, the Faculty is launching an “Innovation Academy” to provide every student with the opportunities and intellectual inspiration to innovate and pursue their engineering passion.
The Innovation Academy is a hub for attracting and cultivating new generation of talents, not only scholars and researchers, but also industry leaders and influencers. It works like an accelerator for inspiration and implementation.
More about Innovat

## 3.9. Generate Answer with GPT-4o-mini

Now we perform the **generation** step:
- Send the context + question to GPT-4o-mini
- Instruct the model to answer **only** using the provided context and to cite sources when possible

In [17]:
messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant for HKU InnoWings and InnoAcademy. "
                "Answer the question using ONLY the provided context documents. "
                "If the answer cannot be found in the context, say: "
                "'I don't have enough information based on the provided documents.' "
                "Be concise, accurate, and professional. Always cite the source URL when possible."
            )
        },
        {
            "role": "user",
            "content": f"Context documents:\n{context}\n\nQuestion: {question}"
        }
    ]

answer = GPT_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    ).choices[0].message.content

Here is the RAG-generated response to the question.

In [18]:
print(answer)

The Innovation Academy is a hub established by the HKU Faculty of Engineering aimed at attracting and cultivating a new generation of talents, including scholars, researchers, industry leaders, and influencers. It serves as an accelerator for inspiration and implementation, providing students with opportunities to innovate and pursue their engineering passions. The academy organizes various programs and activities to encourage a think-out-of-the-box mindset among students, focusing on inspiring, equipping, and showcasing their potential. 

For more information, you can visit the Innovation Academy's page [here](https://www.hku.hk/innoacademy).


## 3.10. Conclusion & Next Steps

**Congratulations!** 🎉  

You have successfully built a complete **Retrieval-Augmented Generation (RAG)** system!

### What you achieved
- Connected to your ChromaDB vector database
- Performed semantic retrieval
- Generated answers with GPT-4o-mini

### Possible Next Steps
1. Implement reranking or hybrid search
2. Evaluate answer quality with different `top_k` values
3. Besides vector search we just implemented, try keyword search, context retrieval, and reranking

You now have the full pipeline: **Scrape → Embed → Retrieve → Generate**.

Well done on completing the Chatbot Challenge data and RAG pipeline!